# LAB CHƯƠNG 1 — TỔNG QUAN VỀ THỊ GIÁC MÁY TÍNH VÀ XỬ LÝ ẢNH

**Nội dung thực hành (theo đề bài Chương 1):**
1. Cài đặt và kiểm tra thư viện **OpenCV** và **Pillow**.
2. Đọc, hiển thị và lưu ảnh với nhiều định dạng khác nhau.
3. Chuyển đổi không gian màu: **RGB → Grayscale**, **HSV** và **LAB**.
4. Cắt xén vùng ảnh (ROI) và thay đổi kích thước theo tỷ lệ / kích thước cố định.
5. Vẽ các hình cơ bản (đường thẳng, hình tròn, hình chữ nhật) và thêm văn bản lên ảnh.

**Ảnh đầu vào:** tự chép ảnh vào thư mục `input/` nằm cạnh notebook. Nếu có nhiều ảnh, chọn tên ảnh tại `IMAGE_FILENAME` ở phần 1.

**Kết quả:** toàn bộ ảnh xử lý, bảng số liệu và biểu đồ được lưu trong thư mục `results/`.

## 0. Cài đặt thư viện và thiết lập

Nếu máy chưa cài OpenCV hoặc Pillow, bỏ dấu `#` ở dòng `%pip install` bên dưới và chạy 1 lần.

In [ ]:
# %pip install opencv-python pillow numpy matplotlib

import cv2
import PIL
from PIL import Image, ImageDraw
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SEED = 42
np.random.seed(SEED)

INPUT_DIR = Path("input")
OUTPUT_DIR = Path("results")
INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("OpenCV:", cv2.__version__)
print("Pillow:", PIL.__version__)
print("NumPy:", np.__version__)

: 

## 1. Đọc và hiển thị ảnh đầu vào

Đặt ảnh trong thư mục `input/`. Nếu chỉ có một ảnh hợp lệ, notebook tự chọn ảnh đó. Nếu có nhiều ảnh, điền tên file vào `IMAGE_FILENAME`.
Notebook đọc ảnh bằng cả **OpenCV** (hệ màu mặc định `BGR`) và **Pillow** (hệ màu `RGB`) để đối chiếu thông số cơ bản.

In [ ]:
IMAGE_FILENAME = "1_cat.jpg"  # Có thể đổi sang "1_anh.png" hoặc None

extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
image_files = sorted(p for p in INPUT_DIR.iterdir()
                     if p.is_file() and p.suffix.lower() in extensions)

if not image_files:
    fallback = [p for p in Path(".").iterdir() if p.is_file() and p.suffix.lower() in extensions]
    if fallback:
        image_files = sorted(fallback)
    else:
        raise FileNotFoundError("Thư mục input/ chưa có ảnh. Hãy chép ảnh của bạn vào input/.")

if IMAGE_FILENAME is None:
    image_path = image_files[0]
else:
    image_path = INPUT_DIR / IMAGE_FILENAME
    if not image_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy ảnh: {image_path}")

image_bgr = cv2.imdecode(np.fromfile(str(image_path), dtype=np.uint8), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise ValueError(f"Không đọc được ảnh bằng OpenCV: {image_path}")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
pil_image = Image.open(image_path).convert("RGB")
height, width, channels = image_bgr.shape

print(f"Ảnh: {image_path.name} | Kích thước: {width} x {height} | Số kênh: {channels} | dtype: {image_bgr.dtype}")
print(f"Pillow mode: {pil_image.mode} | Dung lượng gốc: {image_path.stat().st_size / 1024:.2f} KB")

plt.figure(figsize=(7, 5))
plt.imshow(image_rgb)
plt.title(f"Ảnh đầu vào ({image_path.name} — {width}x{height})")
plt.axis("off")
plt.show()

## Hàm hỗ trợ hiển thị, lưu ảnh và đo chỉ số

Các chỉ số thống kê dùng để đánh giá định lượng qua từng bước xử lý:
- **Mean intensity:** độ sáng trung bình của ảnh.
- **Standard deviation (Std):** độ phân tán mức xám, phản ánh độ tương phản.
- **Laplacian variance:** mức độ sắc nét / chi tiết cạnh của ảnh.
- **MSE & PSNR:** sai số bình phương trung bình và tỷ số tín hiệu trên nhiễu (dB) khi so sánh định dạng nén với ảnh gốc.

In [ ]:
def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img

def show_grid(items, cols=2, figsize=(13, 6), save_name=None):
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, item in zip(axes.flat, items):
        title, img = item[0], item[1]
        cmap = item[2] if len(item) > 2 else None
        is_rgb = item[3] if len(item) > 3 else False
        disp = img if (is_rgb or cmap is not None) else bgr_to_rgb(img)
        ax.imshow(disp, cmap=cmap, vmin=0 if cmap else None, vmax=255 if cmap else None)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    if save_name:
        fig.savefig(OUTPUT_DIR / save_name, dpi=150, bbox_inches="tight")
    plt.show()

def save_img(name, img, params=None):
    path = OUTPUT_DIR / name
    ok, encoded = cv2.imencode(path.suffix or ".png", img, params or [])
    if not ok:
        raise IOError(f"Không lưu được {path}")
    encoded.tofile(str(path))
    return path

def image_metrics(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    return {
        "Mean": float(np.mean(g)),
        "Std": float(np.std(g)),
        "Laplacian Var": float(cv2.Laplacian(g, cv2.CV_64F).var())
    }

def calc_mse_psnr(ref_img, cmp_img):
    mse = float(np.mean((ref_img.astype(np.float64) - cmp_img.astype(np.float64)) ** 2))
    psnr = float("inf") if mse == 0 else float(10.0 * np.log10((255.0 ** 2) / mse))
    return mse, psnr

def plot_metric_bars(labels, values, title, ylabel, save_name=None):
    fig = plt.figure(figsize=(9, 4))
    bars = plt.bar(labels, values)
    plt.title(title, fontweight="bold")
    plt.ylabel(ylabel)
    plt.xticks(rotation=15, ha="right")
    plt.grid(axis="y", alpha=0.25)
    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f"{value:.2f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    if save_name:
        fig.savefig(OUTPUT_DIR / save_name, dpi=150, bbox_inches="tight")
    plt.show()

def show_table(headers, rows, title=None, decimals=2, save_name=None):
    formatted = []
    for row in rows:
        formatted.append([
            ("∞" if isinstance(v, (float, np.floating)) and np.isinf(v)
             else (f"{v:.{decimals}f}" if isinstance(v, (float, np.floating)) else str(v)))
            for v in row
        ])
    fig, ax = plt.subplots(figsize=(max(8, len(headers) * 2.1), 0.7 + 0.44 * (len(rows) + 1)))
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=12, fontweight="bold", pad=12)
    table = ax.table(cellText=formatted, colLabels=headers, loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.5)
    plt.tight_layout()
    if save_name:
        fig.savefig(OUTPUT_DIR / save_name, dpi=150, bbox_inches="tight")
    plt.show()

def show_gray_histogram(named_images, title, save_name=None):
    fig = plt.figure(figsize=(10, 4))
    for label, img in named_images:
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
        hist = cv2.calcHist([g], [0], None, [256], [0, 256]).ravel()
        plt.plot(hist, label=label, linewidth=1.5)
    plt.title(title, fontweight="bold")
    plt.xlabel("Mức xám (0–255)")
    plt.ylabel("Số pixel")
    plt.xlim(0, 255)
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    if save_name:
        fig.savefig(OUTPUT_DIR / save_name, dpi=150, bbox_inches="tight")
    plt.show()

# I. LƯU ẢNH VỚI CÁC ĐỊNH DẠNG KHÁC NHAU

Lưu ảnh đầu vào sang các định dạng phổ biến bằng **OpenCV** (`PNG`, `JPEG`) và **Pillow** (`BMP`, `WEBP`), sau đó so sánh dung lượng tệp và độ suy hao chất lượng (`MSE`, `PSNR`).

In [ ]:
p_png = save_img("01_saved_opencv.png", image_bgr)
p_jpg90 = save_img("01_saved_opencv_q90.jpg", image_bgr, [cv2.IMWRITE_JPEG_QUALITY, 90])
p_jpg40 = save_img("01_saved_opencv_q40.jpg", image_bgr, [cv2.IMWRITE_JPEG_QUALITY, 40])

p_bmp = OUTPUT_DIR / "01_saved_pillow.bmp"
pil_image.save(p_bmp, format="BMP")

p_webp = OUTPUT_DIR / "01_saved_pillow_q80.webp"
pil_image.save(p_webp, format="WEBP", quality=80)

saved_formats = [
    ("BMP (Pillow - Không nén)", p_bmp),
    ("PNG (OpenCV - Lossless)", p_png),
    ("JPG Q=90 (OpenCV)", p_jpg90),
    ("WEBP Q=80 (Pillow)", p_webp),
    ("JPG Q=40 (OpenCV - Nén mạnh)", p_jpg40),
]

bmp_size = p_bmp.stat().st_size
fmt_rows = []
fmt_grid = []
fmt_labels, fmt_sizes_kb = [], []

for label, fpath in saved_formats:
    reloaded = cv2.imdecode(np.fromfile(str(fpath), dtype=np.uint8), cv2.IMREAD_COLOR)
    size_kb = fpath.stat().st_size / 1024.0
    ratio = bmp_size / fpath.stat().st_size
    mse, psnr = calc_mse_psnr(image_bgr, reloaded)
    fmt_rows.append([label, size_kb, ratio, mse, psnr])
    fmt_labels.append(label.split(" (")[0])
    fmt_sizes_kb.append(size_kb)
    if label.startswith(("PNG", "JPG Q=90", "WEBP", "JPG Q=40")):
        fmt_grid.append((f"{label}\n({size_kb:.1f} KB)", reloaded))

show_grid(fmt_grid, cols=2, figsize=(12, 9), save_name="01_format_comparison.png")
show_table(
    ["Định dạng & Thư viện", "Dung lượng (KB)", "Tỷ lệ nén (vs BMP)", "MSE", "PSNR (dB)"],
    fmt_rows,
    title="So sánh dung lượng và chất lượng giữa các định dạng lưu ảnh",
    save_name="01_format_table.png"
)
plot_metric_bars(fmt_labels, fmt_sizes_kb, "So sánh dung lượng tệp theo định dạng", "Dung lượng (KB)",
                 save_name="01_format_sizes.png")

# II. CHUYỂN ĐỔI KHÔNG GIAN MÀU

Thực hiện chuyển đổi từ không gian màu gốc `RGB/BGR` sang **Grayscale**, **HSV** và **LAB**.

### II.1. Chuyển đổi từ RGB sang Grayscale

Đối chiếu 3 cách chuyển đổi:
- **OpenCV:** `cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)`.
- **Pillow:** `pil_image.convert("L")`.
- **Công thức tuyến tính (Slide Chương 1):** $Y = 0.2989R + 0.5870G + 0.1140B$.

In [ ]:
gray_cv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
gray_pil = np.array(pil_image.convert("L"))
gray_formula = np.clip(
    0.2989 * image_rgb[:, :, 0] + 0.5870 * image_rgb[:, :, 1] + 0.1140 * image_rgb[:, :, 2],
    0, 255
).astype(np.uint8)

save_img("02_grayscale_opencv.png", gray_cv)

show_grid([
    ("Ảnh màu gốc (RGB)", image_rgb, None, True),
    ("Grayscale (OpenCV)", gray_cv, "gray"),
    ("Grayscale (Pillow)", gray_pil, "gray"),
    ("Grayscale (Công thức 0.2989R+0.587G+0.114B)", gray_formula, "gray"),
], cols=2, figsize=(12, 9), save_name="02_grayscale_comparison.png")

gray_rows = []
for name, gimg in [
    ("Grayscale (OpenCV)", gray_cv),
    ("Grayscale (Pillow)", gray_pil),
    ("Grayscale (Công thức)", gray_formula),
]:
    m = image_metrics(gimg)
    mae = float(np.mean(np.abs(gray_cv.astype(np.float64) - gimg.astype(np.float64))))
    gray_rows.append([name, m["Mean"], m["Std"], m["Laplacian Var"], mae])

show_table(
    ["Phương pháp", "Độ sáng TB", "Độ tương phản (Std)", "Laplacian Var", "MAE vs OpenCV"],
    gray_rows,
    title="So sánh chỉ số các phương pháp chuyển đổi Grayscale",
    save_name="02_grayscale_table.png"
)
show_gray_histogram([
    ("OpenCV Gray", gray_cv),
    ("Pillow Gray", gray_pil),
    ("Công thức lý thuyết", gray_formula),
], "Phân bố mức xám (Histogram) của ảnh Grayscale", save_name="02_grayscale_hist.png")

### II.2. Chuyển đổi sang không gian màu HSV và LAB

- **HSV (Hue, Saturation, Value):** tách biệt sắc độ (H: `0–179` trong OpenCV), độ bão hòa (S: `0–255`) và độ sáng (V: `0–255`), thuận tiện khi lọc màu theo sắc độ.
- **CIELAB (L, a, b):** tách kênh độ sáng `L` độc lập với hai kênh sắc độ đối lập `a` (xanh lá – đỏ) và `b` (xanh dương – vàng), gần với cảm nhận thị giác người.

In [ ]:
image_hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
image_lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)

save_img("02_anh_HSV.png", image_hsv)
save_img("02_anh_LAB.png", image_lab)

show_grid([
    ("Ảnh gốc (RGB)", image_rgb, None, True),
    ("Không gian màu HSV (hiển thị 3 kênh)", image_hsv, None, True),
    ("Không gian màu LAB (hiển thị 3 kênh)", image_lab, None, True),
], cols=3, figsize=(15, 5), save_name="02_colorspaces_overview.png")

h_ch, s_ch, v_ch = cv2.split(image_hsv)
l_ch, a_ch, b_ch = cv2.split(image_lab)

show_grid([
    ("HSV — Kênh H (Hue / Sắc độ)", h_ch, "hsv"),
    ("HSV — Kênh S (Saturation / Độ bão hòa)", s_ch, "gray"),
    ("HSV — Kênh V (Value / Độ sáng)", v_ch, "gray"),
    ("LAB — Kênh L (Lightness / Độ sáng)", l_ch, "gray"),
    ("LAB — Kênh a (Xanh lá ↔ Đỏ)", a_ch, "gray"),
    ("LAB — Kênh b (Xanh dương ↔ Vàng)", b_ch, "gray"),
], cols=3, figsize=(14, 8), save_name="02_hsv_lab_channels.png")

ch_rows = []
for name, ch in [
    ("RGB - R", image_rgb[:, :, 0]), ("RGB - G", image_rgb[:, :, 1]), ("RGB - B", image_rgb[:, :, 2]),
    ("HSV - H", h_ch), ("HSV - S", s_ch), ("HSV - V", v_ch),
    ("LAB - L", l_ch), ("LAB - a", a_ch), ("LAB - b", b_ch),
]:
    ch_rows.append([name, int(np.min(ch)), int(np.max(ch)), float(np.mean(ch)), float(np.std(ch))])

show_table(
    ["Không gian - Kênh", "Min", "Max", "Trung bình (Mean)", "Độ lệch chuẩn (Std)"],
    ch_rows,
    title="Thống kê giá trị các kênh màu RGB, HSV và LAB",
    save_name="02_channels_stats.png"
)

# III. CẮT XÉN VÀ THAY ĐỔI KÍCH THƯỚC

Thực hiện cắt vùng quan tâm (ROI) và thay đổi kích thước theo tỷ lệ giữ nguyên khung hình (`aspect ratio`) cũng như theo kích thước cố định.

### III.1. Cắt một vùng hình ảnh bất kỳ (Cropping / ROI)

Tọa độ cắt được tính theo tỷ lệ phần trăm kích thước ảnh (`x: 15%→85%`, `y: 10%→65%`) để áp dụng ổn định cho mọi kích thước ảnh đầu vào. Thực hiện song song bằng cắt lát mảng NumPy (OpenCV) và `Image.crop()` (Pillow).

In [ ]:
x1, x2 = int(width * 0.15), int(width * 0.85)
y1, y2 = int(height * 0.10), int(height * 0.65)

roi_preview = image_bgr.copy()
cv2.rectangle(roi_preview, (x1, y1), (x2, y2), (0, 255, 0), max(2, width // 150))
cv2.putText(roi_preview, f"ROI ({x2-x1}x{y2-y1})", (x1, max(25, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

crop_cv = image_bgr[y1:y2, x1:x2]
crop_pil = np.array(pil_image.crop((x1, y1, x2, y2)))

save_img("03_anh_da_cat.png", crop_cv)

show_grid([
    ("Vùng cắt ROI trên ảnh gốc", roi_preview),
    (f"Ảnh đã cắt — OpenCV ({crop_cv.shape[1]}x{crop_cv.shape[0]})", crop_cv),
    (f"Ảnh đã cắt — Pillow ({crop_pil.shape[1]}x{crop_pil.shape[0]})", crop_pil, None, True),
], cols=3, figsize=(14, 5), save_name="03_crop_comparison.png")

m_orig = image_metrics(image_bgr)
m_crop = image_metrics(crop_cv)
show_table(
    ["Ảnh", "Kích thước (W x H)", "Số pixel", "Tỷ lệ diện tích (%)", "Độ sáng TB", "Std"],
    [
        ["Ảnh gốc", f"{width} x {height}", width * height, 100.0, m_orig["Mean"], m_orig["Std"]],
        ["Ảnh sau cắt (ROI)", f"{x2-x1} x {y2-y1}", (x2-x1)*(y2-y1), 100.0*(x2-x1)*(y2-y1)/(width*height), m_crop["Mean"], m_crop["Std"]],
    ],
    title="Thống kê kích thước và đặc trưng trước/sau khi cắt ảnh",
    save_name="03_crop_table.png"
)

### III.2. Thay đổi kích thước theo tỷ lệ và kích thước cố định (Resizing)

- **Thu nhỏ theo tỷ lệ (`0.5x`):** dùng nội suy `cv2.INTER_AREA` (hạn chế răng cưa khi giảm kích thước) và giữ nguyên tỷ lệ khung hình.
- **Phóng to theo tỷ lệ (`1.5x`):** dùng nội suy `cv2.INTER_CUBIC`.
- **Kích thước cố định (`400 x 300`):** ép đúng kích thước mục tiêu (có thể gây méo hình nếu ảnh gốc không cùng tỷ lệ `4:3`).
- **Pillow (`LANCZOS`):** thu nhỏ tỷ lệ `0.5x` để so sánh độ sắc nét.

In [ ]:
scale_down = 0.5
w_down, h_down = int(width * scale_down), int(height * scale_down)
resized_scale_down = cv2.resize(image_bgr, (w_down, h_down), interpolation=cv2.INTER_AREA)

scale_up = 1.5
w_up, h_up = int(width * scale_up), int(height * scale_up)
resized_scale_up = cv2.resize(image_bgr, (w_up, h_up), interpolation=cv2.INTER_CUBIC)

resized_fixed = cv2.resize(image_bgr, (400, 300), interpolation=cv2.INTER_LINEAR)
resized_pil_down = np.array(pil_image.resize((w_down, h_down), Image.Resampling.LANCZOS))

save_img("03_anh_theo_ty_le_05.png", resized_scale_down)
save_img("03_anh_kich_thuoc_co_dinh_400x300.png", resized_fixed)

show_grid([
    (f"Ảnh gốc ({width}x{height})", image_bgr),
    (f"Thu nhỏ tỷ lệ 0.5x - OpenCV ({w_down}x{h_down})", resized_scale_down),
    (f"Kích thước cố định ({400}x{300})", resized_fixed),
    (f"Thu nhỏ tỷ lệ 0.5x - Pillow LANCZOS ({w_down}x{h_down})", resized_pil_down, None, True),
], cols=2, figsize=(12, 9), save_name="03_resize_comparison.png")

resize_items = [
    ("Ảnh gốc", image_bgr),
    ("Tỷ lệ 0.5x (OpenCV AREA)", resized_scale_down),
    ("Tỷ lệ 0.5x (Pillow LANCZOS)", cv2.cvtColor(resized_pil_down, cv2.COLOR_RGB2BGR)),
    ("Tỷ lệ 1.5x (OpenCV CUBIC)", resized_scale_up),
    ("Cố định 400x300 (LINEAR)", resized_fixed),
]
resize_rows, r_labels, r_sharp = [], [], []
for name, rimg in resize_items:
    rh, rw = rimg.shape[:2]
    m = image_metrics(rimg)
    resize_rows.append([name, f"{rw} x {rh}", rw / rh, m["Mean"], m["Std"], m["Laplacian Var"]])
    r_labels.append(name)
    r_sharp.append(m["Laplacian Var"])

show_table(
    ["Phiên bản", "Kích thước (W x H)", "Tỷ lệ W/H", "Độ sáng TB", "Std", "Laplacian Var"],
    resize_rows,
    title="So sánh kích thước, tỷ lệ khung hình và độ sắc nét sau khi Resize",
    save_name="03_resize_table.png"
)
plot_metric_bars(r_labels, r_sharp, "Độ sắc nét (Laplacian Variance) theo phương pháp Resize", "Laplacian Var",
                 save_name="03_resize_sharpness.png")

# IV. VẼ HÌNH CƠ BẢN VÀ THÊM VĂN BẢN

Vẽ các đối tượng hình học cơ bản (**đường thẳng**, **hình tròn**, **hình chữ nhật**) và **chèn văn bản** lên ảnh bằng cả **OpenCV** và **Pillow**.

In [ ]:
thick = max(2, min(width, height) // 100)

# 1. Vẽ đường thẳng bằng OpenCV
img_line = image_bgr.copy()
cv2.line(img_line, (int(width * 0.1), int(height * 0.1)),
         (int(width * 0.9), int(height * 0.9)), (0, 0, 255), thick + 1)
save_img("04_duong_thang.png", img_line)

# 2. Vẽ tổng hợp đường thẳng, hình chữ nhật, hình tròn và văn bản bằng OpenCV
img_draw_cv = image_bgr.copy()
rx1, ry1, rx2, ry2 = int(width * 0.22), int(height * 0.18), int(width * 0.80), int(height * 0.55)
cx, cy, radius = int(width * 0.58), int(height * 0.45), int(min(width, height) * 0.18)

cv2.line(img_draw_cv, (int(width * 0.08), int(height * 0.90)),
         (int(width * 0.92), int(height * 0.90)), (0, 0, 255), thick)
cv2.rectangle(img_draw_cv, (rx1, ry1), (rx2, ry2), (255, 0, 0), thick)
cv2.circle(img_draw_cv, (cx, cy), radius, (0, 255, 0), thick)
cv2.putText(img_draw_cv, "OpenCV: Shapes & Text", (rx1, max(30, ry1 - 12)),
            cv2.FONT_HERSHEY_SIMPLEX, max(0.6, width / 500.0), (0, 255, 255), max(2, thick - 1), cv2.LINE_AA)
save_img("04_ve_hinh_van_ban_opencv.png", img_draw_cv)

# 3. Vẽ hình và thêm văn bản bằng Pillow (ImageDraw)
pil_draw_img = pil_image.copy()
draw = ImageDraw.Draw(pil_draw_img)
draw.line([(int(width * 0.08), int(height * 0.90)), (int(width * 0.92), int(height * 0.90))],
          fill=(255, 0, 0), width=thick)
draw.rectangle([(rx1, ry1), (rx2, ry2)], outline=(0, 128, 255), width=thick)
draw.ellipse([(cx - radius, cy - radius), (cx + radius, cy + radius)], outline=(0, 255, 0), width=thick)
draw.text((rx1, max(10, ry1 - 18)), "Pillow: ImageDraw", fill=(255, 255, 0))
pil_draw_img.save(OUTPUT_DIR / "04_ve_hinh_van_ban_pillow.png")

show_grid([
    ("Ảnh gốc", image_bgr),
    ("Vẽ đường thẳng (OpenCV)", img_line),
    ("Vẽ hình + Văn bản (OpenCV)", img_draw_cv),
    ("Vẽ hình + Văn bản (Pillow)", np.array(pil_draw_img), None, True),
], cols=2, figsize=(12, 9), save_name="04_drawing_comparison.png")

out_files = sorted(p for p in OUTPUT_DIR.iterdir() if p.is_file())
summary_rows = [[p.name, p.suffix.upper().lstrip("."), p.stat().st_size / 1024.0] for p in out_files]
show_table(
    ["Tên tệp trong results/", "Định dạng", "Dung lượng (KB)"],
    summary_rows,
    title=f"Tổng hợp {len(out_files)} tệp kết quả đã xuất trong thư mục results/",
)